In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

import joblib

!ls

 sample_data  'Student_performance_data _.csv'


In [5]:
import pandas as pd

df = pd.read_csv("Student_performance_data _.csv")
df.head()

,StudentID,Age,Gender,Ethnicity,ParentalEducation,StudyTimeWeekly,Absences,Tutoring,ParentalSupport,Extracurricular,Sports,Music,Volunteering,GPA,GradeClass
0,1001,17,1,0,2,19.833723,7,1,2,0,0,1,0,2.929196,2.0
1,1002,18,0,0,1,15.408756,0,0,1,0,0,0,0,3.042915,1.0
2,1003,15,0,2,3,4.210570,26,0,2,0,0,0,0,0.112602,4.0
3,1004,17,1,0,3,10.028829,14,0,3,1,0,0,0,2.054218,3.0
4,1005,17,1,0,2,4.672495,17,1,3,0,0,0,0,1.288061,4.0


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df[['Age',
        'Gender',
        'StudyTimeWeekly',
        'Absences',
        'Tutoring',
        'ParentalSupport',
        'Extracurricular',
        'Sports',
        'Music',
        'Volunteering']]

y = df['GradeClass']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

grade_model = RandomForestClassifier(random_state=42)

grade_model.fit(X_train, y_train)

print("Model trained successfully!")


Model trained successfully!


In [7]:
from sklearn.metrics import accuracy_score
pred = grade_model.predict(X_test)

acc = accuracy_score(y_test, pred)

print("Accuracy =", acc)

Accuracy = 0.6889352818371608


In [8]:
print(grade_model)
type(grade_model)

RandomForestClassifier(random_state=42)


sklearn.ensemble._forest.RandomForestClassifier

In [9]:
import pandas as pd

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': grade_model.feature_importances_
})

importance = importance.sort_values(by='Importance', ascending=False)

print(importance)

           Feature  Importance
3         Absences    0.483537
2  StudyTimeWeekly    0.211384
5  ParentalSupport    0.076475
0              Age    0.065252
1           Gender    0.031903
7           Sports    0.030400
6  Extracurricular    0.030345
4         Tutoring    0.025580
8            Music    0.023867
9     Volunteering    0.021257


In [10]:
X = df[['Age',
        'Gender',
        'StudyTimeWeekly',
        'Absences',
        'Tutoring',
        'ParentalSupport',
        'Extracurricular',
        'Sports',
        'Music',
        'Volunteering',
        'GPA']]

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

y = df['GradeClass']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

grade_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    random_state=42
)

grade_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=15, n_estimators=300, random_state=42)

In [12]:
from sklearn.metrics import accuracy_score

pred = grade_model.predict(X_test)

print("Accuracy =", accuracy_score(y_test, pred))

Accuracy = 0.9144050104384134


In [13]:
import joblib

joblib.dump(grade_model, "grade_class_model.pkl")
print("grade_model saved as grade_class_model.pkl")

grade_model saved as grade_class_model.pkl


In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Note: GPA itself is excluded from the features here since we are predicting GPA
features_gpa = ['Age', 'Gender', 'StudyTimeWeekly', 'Absences', 'Tutoring',
                 'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering']

Xg = df[features_gpa]
yg = df['GPA']

Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    Xg, yg, test_size=0.2, random_state=42
)

gpa_model = LinearRegression()
gpa_model.fit(Xg_train, yg_train)

gpa_pred = gpa_model.predict(Xg_test)

print("GPA Regression MAE :", mean_absolute_error(yg_test, gpa_pred))
print("GPA Regression R2  :", r2_score(yg_test, gpa_pred))

GPA Regression MAE : 0.1552487479513005
GPA Regression R2  : 0.9533443792717046


In [15]:
import joblib

joblib.dump(gpa_model, "gpa_regression_model.pkl")
print("gpa_model saved as gpa_regression_model.pkl")

gpa_model saved as gpa_regression_model.pkl


In [16]:
import numpy as np

np.random.seed(42)

SUBJECTS = ["Mathematics", "Physics", "Computer Science", "English", "History"]

def generate_attendance_log(student_row, n_days=90, subjects=SUBJECTS):
    """
    Simulates a daily, per-subject attendance log for one student.
    The student's real 'Absences' value sets their baseline absence rate,
    so the simulated log stays consistent with the real dataset.
    """
    base_absence_rate = min(student_row['Absences'] / 30, 0.9)
    records = []
    start_date = pd.Timestamp("2025-01-06")
    day = 0
    date = start_date
    while day < n_days:
        if date.weekday() < 5:  # weekdays only
            for subj in subjects:
                jitter = np.random.normal(0, 0.07)
                absence_prob = np.clip(base_absence_rate + jitter, 0.02, 0.95)
                present = np.random.rand() > absence_prob
                records.append({
                    "StudentID": student_row['StudentID'],
                    "Date": date,
                    "Subject": subj,
                    "Present": int(present)
                })
            day += 1
        date += pd.Timedelta(days=1)
    return pd.DataFrame(records)


SUBJECT_DIFFICULTY = {
    "Mathematics": -0.3, "Physics": -0.2, "Computer Science": 0.1,
    "English": 0.2, "History": 0.15
}

def generate_subject_scores(student_row, subjects=SUBJECTS):
    """
    Simulates per-subject scores (0-100) centered on the student's real GPA,
    with subject-specific difficulty offsets and random noise.
    """
    base = student_row['GPA'] / 4 * 100
    scores = {}
    for subj in subjects:
        diff = SUBJECT_DIFFICULTY.get(subj, 0)
        noise = np.random.normal(0, 5)
        score = np.clip(base + diff * 20 + noise, 0, 100)
        scores[subj] = round(score, 1)
    return scores

print("Attendance & subject score simulators ready.")

Attendance & subject score simulators ready.


In [17]:
class AttendanceTracker:
    def __init__(self):
        self.records = pd.DataFrame(columns=["StudentID", "Date", "Subject", "Present"])

    def mark_attendance(self, student_id, date, subject, present):
        """Manually mark a single attendance entry."""
        new_row = {
            "StudentID": student_id,
            "Date": pd.to_datetime(date),
            "Subject": subject,
            "Present": int(present)
        }
        self.records = pd.concat([self.records, pd.DataFrame([new_row])], ignore_index=True)

    def load_log(self, log_df):
        """Bulk-load an attendance log (e.g. from generate_attendance_log)."""
        self.records = pd.concat([self.records, log_df], ignore_index=True)

    def attendance_percentage(self, student_id, subject=None):
        recs = self.records[self.records["StudentID"] == student_id]
        if subject:
            recs = recs[recs["Subject"] == subject]
        if len(recs) == 0:
            return None
        return round(recs["Present"].mean() * 100, 2)

    def subject_breakdown(self, student_id):
        recs = self.records[self.records["StudentID"] == student_id]
        return recs.groupby("Subject")["Present"].mean().mul(100).round(2).to_dict()

    def total_absences(self, student_id):
        recs = self.records[self.records["StudentID"] == student_id]
        return int((recs["Present"] == 0).sum())


# Quick test with one student
tracker_demo = AttendanceTracker()
demo_log = generate_attendance_log(df.iloc[0])
tracker_demo.load_log(demo_log)

print("Overall attendance %:", tracker_demo.attendance_percentage(df.iloc[0]['StudentID']))
print("Per-subject breakdown:", tracker_demo.subject_breakdown(df.iloc[0]['StudentID']))
print("Total absences logged:", tracker_demo.total_absences(df.iloc[0]['StudentID']))

Overall attendance %: 74.0
Per-subject breakdown: {'Computer Science': 64.44444444444444, 'English': 83.33333333333334, 'History': 68.88888888888889, 'Mathematics': 76.66666666666667, 'Physics': 76.66666666666667}
Total absences logged: 117


/tmp/ipykernel_1097/240520815.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.records = pd.concat([self.records, log_df], ignore_index=True)


In [18]:
from sklearn.ensemble import RandomForestRegressor

sample_n = 400
sim_students = df.sample(sample_n, random_state=1).reset_index(drop=True)

rows = []
for _, srow in sim_students.iterrows():
    log = generate_attendance_log(srow, n_days=60)
    att_pct = log['Present'].mean() * 100
    rows.append({
        "StudyTimeWeekly": srow['StudyTimeWeekly'],
        "Tutoring": srow['Tutoring'],
        "ParentalSupport": srow['ParentalSupport'],
        "Extracurricular": srow['Extracurricular'],
        "PastAbsences": srow['Absences'],
        "GPA": srow['GPA'],
        "FutureAttendancePct": att_pct
    })

att_train_df = pd.DataFrame(rows)

Xa = att_train_df.drop(columns=["FutureAttendancePct"])
ya = att_train_df["FutureAttendancePct"]

Xa_train, Xa_test, ya_train, ya_test = train_test_split(Xa, ya, test_size=0.2, random_state=42)

attendance_model = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
attendance_model.fit(Xa_train, ya_train)

ya_pred = attendance_model.predict(Xa_test)
print("Attendance Prediction MAE :", mean_absolute_error(ya_test, ya_pred))
print("Attendance Prediction R2  :", r2_score(ya_test, ya_pred))

joblib.dump(attendance_model, "attendance_prediction_model.pkl")
print("attendance_model saved as attendance_prediction_model.pkl")

Attendance Prediction MAE : 1.9572442346611063
Attendance Prediction R2  : 0.9926678823129772
attendance_model saved as attendance_prediction_model.pkl


In [19]:
class SubjectAnalytics:
    def __init__(self, tracker):
        self.tracker = tracker
        self.scores = {}

    def add_scores(self, student_id, scores_dict):
        self.scores[student_id] = scores_dict

    def report(self, student_id):
        att = self.tracker.subject_breakdown(student_id)
        scr = self.scores.get(student_id, {})
        subjects = set(att.keys()) | set(scr.keys())
        report = {}
        for s in subjects:
            report[s] = {"AttendancePct": att.get(s), "Score": scr.get(s)}
        return report

    def weakest_subject(self, student_id):
        scr = self.scores.get(student_id, {})
        if not scr:
            return None
        return min(scr, key=scr.get)

    def strongest_subject(self, student_id):
        scr = self.scores.get(student_id, {})
        if not scr:
            return None
        return max(scr, key=scr.get)


# Quick test
analytics_demo = SubjectAnalytics(tracker_demo)
analytics_demo.add_scores(df.iloc[0]['StudentID'], generate_subject_scores(df.iloc[0]))

print("Subject report:", analytics_demo.report(df.iloc[0]['StudentID']))
print("Weakest subject :", analytics_demo.weakest_subject(df.iloc[0]['StudentID']))
print("Strongest subject:", analytics_demo.strongest_subject(df.iloc[0]['StudentID']))

Subject report: {'History': {'AttendancePct': 68.88888888888889, 'Score': np.float64(82.9)}, 'Mathematics': {'AttendancePct': 76.66666666666667, 'Score': np.float64(63.6)}, 'Computer Science': {'AttendancePct': 64.44444444444444, 'Score': np.float64(74.2)}, 'English': {'AttendancePct': 83.33333333333334, 'Score': np.float64(76.2)}, 'Physics': {'AttendancePct': 76.66666666666667, 'Score': np.float64(68.8)}}
Weakest subject : Mathematics
Strongest subject: History


In [20]:
def recommend_study_plan(student_id, gpa, predicted_gpa, attendance_pct,
                          weakest_subject, study_time_weekly):
    recs = []

    if attendance_pct is not None and attendance_pct < 75:
        recs.append(
            f"Attendance is {attendance_pct}%, below the 75% safe threshold. "
            f"Prioritize regular attendance, especially in {weakest_subject}."
        )

    if predicted_gpa < gpa - 0.2:
        recs.append(
            "Predicted GPA trend is declining. Consider revisiting recent topics "
            "and increasing weekly study time."
        )

    if study_time_weekly < 10:
        recs.append(
            f"Weekly study time is only {round(study_time_weekly, 1)} hrs. "
            "Aim for at least 10-12 hrs/week for steady improvement."
        )

    if weakest_subject:
        recs.append(f"Focus extra revision time on {weakest_subject}, your weakest subject.")

    if gpa >= 3.5 and (attendance_pct is None or attendance_pct >= 85):
        recs.append(
            "Strong performance overall — maintain current routine and consider "
            "peer tutoring or advanced material."
        )

    if not recs:
        recs.append("Performance looks stable. Keep up consistent study habits.")

    return recs


# Quick test
sample_recs = recommend_study_plan(
    student_id=df.iloc[0]['StudentID'],
    gpa=df.iloc[0]['GPA'],
    predicted_gpa=gpa_model.predict(df.iloc[[0]][features_gpa])[0],
    attendance_pct=tracker_demo.attendance_percentage(df.iloc[0]['StudentID']),
    weakest_subject=analytics_demo.weakest_subject(df.iloc[0]['StudentID']),
    study_time_weekly=df.iloc[0]['StudyTimeWeekly']
)
for r in sample_recs:
    print("-", r)

- Attendance is 74.0%, below the 75% safe threshold. Prioritize regular attendance, especially in Mathematics.
- Focus extra revision time on Mathematics, your weakest subject.


In [21]:
class StudentPortal:
    def __init__(self, df, grade_model, gpa_model, attendance_model):
        self.df = df
        self.grade_model = grade_model
        self.gpa_model = gpa_model
        self.attendance_model = attendance_model
        self.tracker = AttendanceTracker()
        self.analytics = SubjectAnalytics(self.tracker)
        self._init_demo_data()

    def _init_demo_data(self):
        """Seeds the tracker and analytics with simulated attendance/subject data
        for every student currently loaded, since the source CSV has no log of its own."""
        for _, srow in self.df.iterrows():
            log = generate_attendance_log(srow, n_days=60)
            self.tracker.load_log(log)
            self.analytics.add_scores(srow['StudentID'], generate_subject_scores(srow))

    def get_student(self, student_id):
        row = self.df[self.df['StudentID'] == student_id]
        if row.empty:
            raise ValueError(f"Student {student_id} not found")
        return row.iloc[0]

    def predict_grade_class(self, student_id):
        srow = self.get_student(student_id)
        feats = srow[['Age', 'Gender', 'StudyTimeWeekly', 'Absences', 'Tutoring',
                       'ParentalSupport', 'Extracurricular', 'Sports', 'Music',
                       'Volunteering', 'GPA']].to_frame().T
        return self.grade_model.predict(feats)[0]

    def predict_gpa(self, student_id):
        srow = self.get_student(student_id)
        feats = srow[features_gpa].to_frame().T
        return round(self.gpa_model.predict(feats)[0], 2)

    def predict_attendance(self, student_id):
        srow = self.get_student(student_id)
        feats = pd.DataFrame([{
            "StudyTimeWeekly": srow['StudyTimeWeekly'],
            "Tutoring": srow['Tutoring'],
            "ParentalSupport": srow['ParentalSupport'],
            "Extracurricular": srow['Extracurricular'],
            "PastAbsences": srow['Absences'],
            "GPA": srow['GPA']
        }])
        return round(self.attendance_model.predict(feats)[0], 2)

    def subject_report(self, student_id):
        return self.analytics.report(student_id)

    def study_recommendations(self, student_id):
        srow = self.get_student(student_id)
        return recommend_study_plan(
            student_id=student_id,
            gpa=srow['GPA'],
            predicted_gpa=self.predict_gpa(student_id),
            attendance_pct=self.tracker.attendance_percentage(student_id),
            weakest_subject=self.analytics.weakest_subject(student_id),
            study_time_weekly=srow['StudyTimeWeekly']
        )

    def dashboard(self, student_id):
        srow = self.get_student(student_id)
        current_gpa = srow['GPA']
        current_grade_class = srow['GradeClass']
        print(f"===== Dashboard: Student {student_id} =====")
        print(f"Current GPA: {current_gpa:.2f}  |  Predicted GPA: {self.predict_gpa(student_id)}")
        print(f"Current GradeClass: {current_grade_class}  |  "
              f"Predicted GradeClass: {self.predict_grade_class(student_id)}")
        print(f"Attendance so far: {self.tracker.attendance_percentage(student_id)}%  |  "
              f"Predicted future attendance: {self.predict_attendance(student_id)}%")
        print("Subject Report:")
        for subj, vals in self.subject_report(student_id).items():
            att_pct = vals['AttendancePct']
            score = vals['Score']
            print(f"  - {subj}: Attendance {att_pct}%, Score {score}")
        print("Study Recommendations:")
        for r in self.study_recommendations(student_id):
            print("  *", r)

In [22]:
portal = StudentPortal(df.head(50), grade_model, gpa_model, attendance_model)

# Run the dashboard for a sample student
portal.dashboard(df.iloc[0]['StudentID'])

/tmp/ipykernel_1097/240520815.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.records = pd.concat([self.records, log_df], ignore_index=True)


===== Dashboard: Student 1001.0 =====
Current GPA: 2.93  |  Predicted GPA: 3.11
Current GradeClass: 2.0  |  Predicted GradeClass: 2.0
Attendance so far: 78.33%  |  Predicted future attendance: 77.51%
Subject Report:
  - History: Attendance 90.0%, Score 67.8
  - Mathematics: Attendance 76.66666666666667%, Score 66.0
  - Computer Science: Attendance 86.66666666666667%, Score 77.8
  - English: Attendance 75.0%, Score 77.7
  - Physics: Attendance 63.33333333333333%, Score 63.9
Study Recommendations:
  * Focus extra revision time on Physics, your weakest subject.


In [23]:
portal.dashboard(df.iloc[10]['StudentID'])

===== Dashboard: Student 1011.0 =====
Current GPA: 2.15  |  Predicted GPA: 1.91
Current GradeClass: 3.0  |  Predicted GradeClass: 3.0
Attendance so far: 68.67%  |  Predicted future attendance: 64.83%
Subject Report:
  - History: Attendance 75.0%, Score 59.6
  - Mathematics: Attendance 66.66666666666666%, Score 48.1
  - Computer Science: Attendance 66.66666666666666%, Score 58.2
  - English: Attendance 61.66666666666667%, Score 56.9
  - Physics: Attendance 73.33333333333333%, Score 52.7
Study Recommendations:
  * Attendance is 68.67%, below the 75% safe threshold. Prioritize regular attendance, especially in Mathematics.
  * Predicted GPA trend is declining. Consider revisiting recent topics and increasing weekly study time.
  * Focus extra revision time on Mathematics, your weakest subject.
